# cancer-recon-apoptosis — Step 1: Boltz-2 Oracle Smoke Test (Colab)

**Goal:** verify Boltz-2 can separate a real DR5 binder from non-binders, so it can serve as the design-loop oracle.

**Two-step design:**
- **Cell 6 — predict** (`scripts/01`): Boltz predicts 3 complexes of DR5 ECD + binder:
  `positive` (TRAIL ectodomain, real binder), `negative` (scrambled TRAIL, unfoldable), `negative_folded` (lysozyme, well-folded non-binder).
- **Cell 8 — decide** (`scripts/02`): computes the interface metrics from the saved outputs and decides.

**Why a two-axis filter (not any single score):** no single metric separates the real binder from *both* decoys.
- raw **ipTM** is fooled by the scrambled decoy (it scored *higher*);
- **pDockQ / interface pLDDT** are fooled by the folded lysozyme (Boltz confidently docks a non-binder → as high as the real binder);
- but **interface pLDDT AND inter-chain PAE together** work: the scrambled decoy fails the pLDDT axis, lysozyme fails the PAE axis, and only the real binder passes both.

**PASS:** positive satisfies `interface pLDDT ≥ 0.70` AND `inter-chain PAE ≤ 15 Å` (standard AF-Multimer thresholds), and **no** non-binder does.

**Caveat (in the code):** n=1 per class — the *mechanism* is standard, but thresholds need calibration on a benchmark before becoming the RL reward (likely pDockQ2 or ipSAE).

**Required runtime:** Colab → Runtime → Change runtime type → **T4 GPU**. Free tier works.

**Idempotency:** every cell is safe to re-run; each complex caches in `runs/step1_boltz/<name>/` and is skipped if already predicted. Cells print `[CELL N] ▶ … ✓`.

## 1. Verify GPU

If `nvidia-smi` shows no GPU, go to *Runtime → Change runtime type → T4 GPU* and rerun.

In [ ]:
#@title Cell 1 — verify GPU
print("[CELL 1] ▶ nvidia-smi")
!nvidia-smi || echo "[CELL 1] ✗ no GPU — change runtime to T4"
print("[CELL 1] ✓ done")

In [ ]:
#@title Cell 2 — verify torch + CUDA
print("[CELL 2] ▶ check torch")
try:
    import torch
    print(f"  torch={torch.__version__}  cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  device={torch.cuda.get_device_name(0)}")
        print(f"  VRAM={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        print("[CELL 2] ✓ done")
    else:
        print("[CELL 2] ✗ no CUDA — runtime is CPU-only; Cell 4 (boltz) will fail")
except Exception as e:
    print(f"[CELL 2] ✗ {type(e).__name__}: {e}")

## 2. Configure repo URL and clone

Set `REPO_URL` to your GitHub repo once you've pushed it. Default below uses the canonical `AnshumanAtrey/cancer-recon-apoptosis` — change if your username is different.

In [ ]:
#@title Cell 3 — clone / pull repo (idempotent)
print("[CELL 3] ▶ clone or pull repo")
import os
from pathlib import Path

# ← EDIT if your GitHub repo is at a different URL
REPO_URL = "https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git"
REPO_DIR = Path("/content/cancer-recon-apoptosis")

try:
    if REPO_DIR.exists():
        print(f"  repo already cloned at {REPO_DIR} — pulling latest")
        !cd {REPO_DIR} && git pull --ff-only
    else:
        print(f"  cloning {REPO_URL}")
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    print(f"  cwd = {os.getcwd()}")
    assert (REPO_DIR / "scripts" / "01_boltz_smoketest.py").exists(), "smoke test script missing"
    print("[CELL 3] ✓ done")
except Exception as e:
    print(f"[CELL 3] ✗ {type(e).__name__}: {e}")
    raise

## 3. Install Boltz-2

First-time install pulls ~10 GB of model weights on first inference (downloaded lazily). The `pip install` itself is fast (~2 min).

In [ ]:
#@title Cell 4 — install Boltz-2 (idempotent — pip skips if already installed)
print("[CELL 4] ▶ pip install boltz biopython")
import shutil, importlib.util
already = shutil.which("boltz") is not None and importlib.util.find_spec("Bio") is not None
if already:
    print("  boltz + biopython already installed — skipping")
else:
    !pip install --quiet boltz biopython
print(f"  boltz CLI: {shutil.which('boltz')}")
print("[CELL 4] ✓ done")

In [ ]:
#@title Cell 5 — verify boltz CLI works
print("[CELL 5] ▶ boltz --help")
import shutil
if shutil.which("boltz") is None:
    print("[CELL 5] ✗ boltz CLI not in PATH — restart runtime and re-run Cell 4")
else:
    !boltz --help | head -15
    print("[CELL 5] ✓ done")

## 4. Predict the complexes (Cell 6)

`scripts/01_boltz_smoketest.py` predicts three DR5–binder complexes with Boltz-2 and saves the raw confidence outputs. It does **not** decide pass/fail — that's Cell 8.

1. `positive` — DR5 ECD + TRAIL ectodomain (residues 114–281). Real DR5 binder.
2. `negative` — DR5 ECD + scrambled TRAIL ectodomain. Composition-matched, unfoldable.
3. `negative_folded` — DR5 ECD + hen lysozyme. A **well-folded non-binder** — the rigorous control (isolates interface specificity from mere foldability).

Each complex: MSA fetch (ColabFold server) → structure prediction. ~3 min each on T4 after the one-time ~10 GB weight download. Already-predicted complexes are skipped.

In [ ]:
#@title Cell 6 — predict complexes (resumable: caches per complex)
# Predicts positive / negative / negative_folded. Already-done complexes are skipped.
# To force a full rerun:  !rm -rf runs/step1_boltz
print("[CELL 6] ▶ python -u scripts/01_boltz_smoketest.py")
import subprocess, sys
# Popen+PIPE+`python -u`: subprocess.call inherits the OS stdout fd which Colab's
# kernel doesn't capture; re-printing each line via print() makes logs stream live.
proc = subprocess.Popen([sys.executable, "-u", "scripts/01_boltz_smoketest.py"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
exit_code = proc.wait()
print(f"\n[CELL 6] script exit code = {exit_code}")
if exit_code == 0:
    print("[CELL 6] ✓ all complexes predicted. Run Cell 7 (inspect) then Cell 8 (decide).")
elif exit_code == 2:
    print("[CELL 6] ✗ boltz CLI missing. Re-run Cell 4, restart runtime if needed.")
elif exit_code == 3:
    print("[CELL 6] ✗ a complex produced no confidence JSON — read the boltz.log tail above; re-run this cell.")
elif exit_code == 4:
    print("[CELL 6] ✗ FASTA missing — verify data/sequences/ was cloned (need dr5/trail/scrambled/lysozyme).")
else:
    print(f"[CELL 6] ✗ unknown exit code {exit_code} — check log above")

## 5. Inspect + Decide (Cells 7–8)

**Cell 7** (optional) dumps the raw confidence fields per complex so you can see the numbers yourself.

**Cell 8** runs `scripts/02_interface_metrics.py` — the real verdict. It re-analyses the saved structures (no Boltz recompute), computes **pDockQ**, **interface pLDDT**, and **interface PAE**, prints a side-by-side panel, and decides PASS/FAIL on pDockQ margin + interface pLDDT. This is fast (seconds) and can be re-run any time on existing outputs.

In [ ]:
#@title Cell 7 — inspect raw confidence outputs (optional)
print("[CELL 7] ▶ list outputs + display confidence JSONs")
import json
from pathlib import Path

RUN_DIR = Path("runs/step1_boltz")
if not RUN_DIR.exists():
    print(f"[CELL 7] ✗ {RUN_DIR} does not exist — re-run Cell 6")
else:
    print("=== Output tree (one level) ===")
    !ls runs/step1_boltz
    print("\n=== Confidence JSONs ===")
    found = list(RUN_DIR.rglob("confidence_*.json"))
    if not found:
        print("  no confidence JSONs found — read the boltz.log tail from Cell 6")
    else:
        for j in sorted(found):
            d = json.loads(j.read_text())
            who = j.relative_to(RUN_DIR).parts[0]
            print(f"\n--- {who} ---")
            print(f"  iptm           = {d.get('iptm')}")
            print(f"  ptm            = {d.get('ptm')}")
            print(f"  complex_plddt  = {d.get('complex_plddt')}")
            print(f"  complex_iplddt = {d.get('complex_iplddt')}   (interface pLDDT)")
            print(f"  complex_ipde   = {d.get('complex_ipde')}")
            print(f"  confidence     = {d.get('confidence_score')}")
    print("\n[CELL 7] ✓ done  (note: raw ipTM is NOT the verdict — Cell 8 decides)")

In [ ]:
#@title Cell 8 — DECIDE: interface metrics (pDockQ / interface pLDDT / interface PAE)
# Re-analyses saved Boltz outputs — NO recompute. Safe to re-run anytime.
print("[CELL 8] ▶ python -u scripts/02_interface_metrics.py")
import subprocess, sys
proc = subprocess.Popen([sys.executable, "-u", "scripts/02_interface_metrics.py"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
exit_code = proc.wait()
print(f"\n[CELL 8] exit code = {exit_code}")
if exit_code == 0:
    print("[CELL 8] ✅ PASS — oracle separates binder from non-binders. Proceed to Step 2.")
elif exit_code == 1:
    print("[CELL 8] ❌ FAIL — see reasons + pivot options above (TRAIL trimer / AF3 / best-of-N / pDockQ2 / ipSAE).")
elif exit_code == 2:
    print("[CELL 8] ✗ missing dependency — run: !pip install -q numpy biopython")
elif exit_code == 3:
    print("[CELL 8] ✗ no predictions found — run Cell 6 first.")
else:
    print(f"[CELL 8] ✗ unknown exit code {exit_code}")

## 6. Save outputs to Google Drive (optional)

Colab sessions are ephemeral — when the runtime resets, `/content/` is wiped. To persist the smoke test results, mount Drive and copy.

In [ ]:
#@title Cell 9 — persist results to Google Drive (optional, idempotent)
print("[CELL 9] ▶ mount Drive + copy runs/step1_boltz/")
from pathlib import Path
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE_OUT = Path('/content/drive/MyDrive/cancer-recon-apoptosis/step1_boltz')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    !cp -r runs/step1_boltz/. '{DRIVE_OUT}/' 2>&1 | tail -5
    print(f"  saved → {DRIVE_OUT}")
    !ls -la '{DRIVE_OUT}' | head -10
    print("[CELL 9] ✓ done")
except ModuleNotFoundError:
    print("[CELL 9] skipped — not running in Colab (no google.colab module)")
except Exception as e:
    print(f"[CELL 9] ✗ {type(e).__name__}: {e}")

## 7. Optional — push results back to GitHub

If you want results committed back to the repo, run the cell below. **Skip if you don't want runs/ in version control** (the project's `.gitignore` already excludes `runs/`).

In [ ]:
# ⚠️ Only run if you want to commit smoke-test results to the repo.
# `.gitignore` excludes runs/ by default — uncomment if you want to override.

# !git config user.name 'Anshuman Atrey'
# !git config user.email 'YOUR_EMAIL@example.com'
# !git add -f runs/step1_boltz/  # -f overrides .gitignore
# !git commit -m 'step 1: Boltz-2 smoke test results'
# !git push  # will prompt for GitHub PAT
print("  (skipped — uncomment cell to push)")

---

## If the smoke test PASSED

1. Update `runs/step1_local/SMOKETEST_RESULTS.md` with the real cloud numbers.
2. Proceed to **Step 2** — CellChat cancer-restricted target shortlist (`scripts/02_cellchat_targets.R`, to be written).

## If the smoke test FAILED (gap < 2 kcal/mol)

See `ASSESSMENT.md` Day-1 kill criteria. Pivot options:
1. **AlphaFold 3 Server** — submit complexes via web API for cross-check.
2. **ABFE pipeline** — Recursion Oct 2025 pipeline atop Boltz-2 ensemble.
3. **OpenMM MD refinement** — 5 ns MD after Boltz-2 prediction for refined ΔG.

## Common Colab issues

- **`boltz: command not found`** → restart runtime after `pip install` and re-run cells from §3.
- **OOM error** → switch to High-RAM runtime, or use `--diffusion_samples 1` instead of 5.
- **MSA server timeout** → re-run; ColabFold MSA queue is sometimes congested. Or pre-compute MSAs locally and pass via `--msa_dir`.
- **Runtime disconnected** → free tier idle limit. Keep tab focused, or upgrade to Colab Pro ($10/mo) for A100.